In [1]:
import pandas as pd
import numpy as np

# ==========================================
# Phase 1: Data Preparation & Enrichment
# ==========================================

# 1. Load the raw dataset
file_path = '../data/raw/retail_sales.csv'
df = pd.read_csv(file_path)

df.head()

,date,store_id,item_id,sales,price,promo,weekday,month
0,2019-01-01,store_1,item_1,41,21.30,0,1,1
1,2019-01-02,store_1,item_1,53,21.30,0,2,1
2,2019-01-03,store_1,item_1,39,21.30,0,3,1
3,2019-01-04,store_1,item_1,35,21.30,0,4,1
4,2019-01-05,store_1,item_1,51,17.04,1,5,1


In [2]:
# Look at 15 completely random rows from anywhere in the 4.5 million rows
display(df.sample(15))

,date,store_id,item_id,sales,price,promo,weekday,month
1538081,2020-08-12,store_17,item_43,32,89.78,0,2,8
314979,2021-06-26,store_4,item_23,16,27.44,0,5,6
2313636,2019-04-05,store_26,item_18,50,69.75,0,4,4
654097,2020-01-25,store_8,item_9,8,97.31,0,5,1
2269889,2019-06-21,store_25,item_44,22,68.75,0,4,6
4421457,2020-12-12,store_49,item_22,12,23.65,0,5,12
238439,2021-11-25,store_3,item_31,60,41.38,1,3,11
827463,2019-10-13,store_10,item_4,26,43.18,0,6,10
4010664,2021-02-07,store_44,item_47,36,89.53,0,6,2
1833878,2020-07-28,store_21,item_5,31,23.91,0,1,7


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4565000 entries, 0 to 4564999
Data columns (total 8 columns):
 #   Column    Dtype  
---  ------    -----  
 0   date      str    
 1   store_id  str    
 2   item_id   str    
 3   sales     int64  
 4   price     float64
 5   promo     int64  
 6   weekday   int64  
 7   month     int64  
dtypes: float64(1), int64(4), str(3)
memory usage: 278.6 MB


In [4]:
df.isna().sum()

date        0
store_id    0
item_id     0
sales       0
price       0
promo       0
weekday     0
month       0
dtype: int64

In [5]:
df.store_id.unique()

<StringArray>
[ 'store_1',  'store_2',  'store_3',  'store_4',  'store_5',  'store_6',
  'store_7',  'store_8',  'store_9', 'store_10', 'store_11', 'store_12',
 'store_13', 'store_14', 'store_15', 'store_16', 'store_17', 'store_18',
 'store_19', 'store_20', 'store_21', 'store_22', 'store_23', 'store_24',
 'store_25', 'store_26', 'store_27', 'store_28', 'store_29', 'store_30',
 'store_31', 'store_32', 'store_33', 'store_34', 'store_35', 'store_36',
 'store_37', 'store_38', 'store_39', 'store_40', 'store_41', 'store_42',
 'store_43', 'store_44', 'store_45', 'store_46', 'store_47', 'store_48',
 'store_49', 'store_50']
Length: 50, dtype: str

In [6]:
df['date'] = pd.to_datetime(df['date'])

In [7]:
# Instead of randomizing rows, we assign fixed locations to specific stores.
unique_stores = df['store_id'].unique()
countries = ['Nigeria', 'Kenya', 'South Africa', 'Ghana', 'Egypt']

In [8]:
# Let's map stores to countries and assign City Tiers based on a rule:
# We will make half the stores Tier 1, half Tier 2.
np.random.seed(42)
store_mapping_temp = pd.DataFrame({
    'old_store_id': unique_stores,
    'country': np.random.choice(countries, size=len(unique_stores)),  # What this code does is that it assigns one of the countries in our list called countries randomly to a store. The second argument ensures that it does this for the whole rows in our dataset. It is thereby filling every unique store with a country.
    'city_tier': np.random.choice(['Tier 1 (Major)', 'Tier 2 (Mid-level)'], size=len(unique_stores))
})

In [9]:
# Generate new store IDs based on (country, city_tier) combinations
# For each unique location, create ~50 stores with seeded noise
np.random.seed(42)
city_tiers = ['Tier 1 (Major)', 'Tier 2 (Mid-level)']
new_store_mapping = []
store_counter = 1

for country in countries:
    for tier in city_tiers:
        # Add seeded noise to the count (between 45 and 55 around 50)
        num_stores = 50 + np.random.randint(-5, 6)  # -5 to +5 gives us 45-55 range

        for i in range(num_stores):
            new_store_mapping.append({
                'new_store_id': store_counter,
                'country': country,
                'city_tier': tier
            })
            store_counter += 1

store_location_mapping = pd.DataFrame(new_store_mapping)

In [10]:
# Map old stores to new location-based stores
# For each old store, assign it to a random new store in its country+tier combination
old_to_new_mapping = []

for _, row in store_mapping_temp.iterrows():
    country = row['country']
    tier = row['city_tier']

    # Get all new stores for this country+tier combination
    available_new_stores = store_location_mapping[
        (store_location_mapping['country'] == country) &
        (store_location_mapping['city_tier'] == tier)
    ]['new_store_id'].values

    # Randomly assign this old store to one of the new stores
    new_store = np.random.choice(available_new_stores)
    old_to_new_mapping.append({
        'old_store_id': row['old_store_id'],
        'new_store_id': new_store
    })

old_to_new_df = pd.DataFrame(old_to_new_mapping)

# Merge to get the final store mapping with country and city_tier
# First merge old to new mapping with location mapping
store_mapping = old_to_new_df.merge(store_location_mapping, on='new_store_id', how='left')

# Now rename old_store_id to store_id for merging with original df
store_mapping_for_merge = store_mapping.copy()
store_mapping_for_merge = store_mapping_for_merge.rename(columns={'old_store_id': 'store_id'})
store_mapping_for_merge = store_mapping_for_merge.drop(columns=['new_store_id'])

In [11]:
# Data Enrichment: Item Mapping
# ==========================================
# Assign fixed categories, subcategories, and base prices to all 50 items
unique_items = df['item_id'].unique()
product_hierarchy = {
    'Electronics': ['Phones', 'Laptops', 'Accessories'],
    'Fashion': ['Shoes', 'Clothing'],
    'Home': ['Furniture', 'Appliances']
}

In [12]:
price_base = {
    'Phones': (200, 800), 'Laptops': (400, 1500), 'Accessories': (10, 50),
    'Shoes': (30, 120), 'Clothing': (15, 80),
    'Furniture': (100, 600), 'Appliances': (50, 400)
}

In [13]:
categories = list(product_hierarchy.keys())
item_mapping = pd.DataFrame({'item_id': unique_items})
item_mapping['category'] = np.random.choice(categories, size=len(unique_items))
item_mapping['subcategory'] = item_mapping['category'].apply(lambda x: np.random.choice(product_hierarchy[x]))

In [14]:
def generate_base_price(subcat):
    min_price, max_price = price_base[subcat]
    return round(np.random.uniform(min_price, max_price), 2)

item_mapping['base_price'] = item_mapping['subcategory'].apply(generate_base_price)

### Explanation of the above code:


What the code above does is that it takes in an argument which is subcategory. Then it takes price_base which is a dictionary that we created earlier and it looks up the subcategory in the price_base dictionary to get the minimum and maximum price for that subcategory. Then it generates a random price between the minimum and maximum price using np.random.uniform and rounds it to 2 decimal places. Finally, it returns the generated base price for that subcategory.

In [15]:
# This joins our logical locations and products to the 4.5 million transactions
df = df.merge(store_mapping_for_merge, on='store_id', how='left')
df = df.merge(item_mapping, on='item_id', how='left')

In [16]:
# Feature Engineering

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

In [17]:
# Rename 'sales' to 'demand' to fit our architecture blueprint
df = df.rename(columns={'sales': 'demand'})

In [20]:
# ==========================================
# Demand Distribution Engineering
# ==========================================
# Adjust demand so Tier 1 (Major) areas have 2.5x more demand than Tier 2 (Mid-level)
# This reflects realistic market behavior: urban areas have higher volume

np.random.seed(42)

# VECTORIZED APPROACH - Much faster than apply()
# For Tier 1 (Major) - Higher base demand (2.5x)
# For Tier 2 (Mid-level) - Lower base demand (1.0x baseline)
tier1_mask = df['city_tier'] == 'Tier 1 (Major)'
df['demand'] = df['demand'].where(~tier1_mask, df['demand'] * 2.5).astype(int)


In [21]:
# ==========================================
# Price Engineering Based on Demand
# ==========================================
# Create a realistic correlation between price and demand:
# Higher demand products generally have strategic pricing (premium or competitive)
# We add noise to make it realistic (not perfectly correlated)

np.random.seed(42)

# VECTORIZED APPROACH - Replaces slow apply(axis=1) with NumPy vectorization
# Logic:
# - High demand items: Usually priced strategically (premium pricing, 1.0-1.5x base)
# - Low demand items: Competitive pricing to move inventory (0.8-1.0x base)
# - Noise ensures realistic variation (not purely synthetic)

# Normalize demand to a 0-1 scale for calculation
# (demand varies roughly from 1-200+ in engineered data)
normalized_demand = np.minimum(df['demand'].values / 150.0, 1.0)

# Price adjustment based on demand (vectorized)
# High demand → Premium pricing strategy (1.0 to 1.5x base_price)
# Low demand → Competitive pricing strategy (0.8 to 1.0x base_price)
demand_price_factor = 0.8 + (normalized_demand * 0.7)  # Range: 0.8 to 1.5

# Generate reproducible noise efficiently (set seed once, generate all noise at once)
# This replaces the 4.5M np.random.seed() calls with a single seed call
np.random.seed(42)
noise_array = np.random.uniform(0.9, 1.1, size=len(df))

# Vectorized final price calculation (100x faster than row-by-row apply)
df['price'] = (df['base_price'].values * demand_price_factor * noise_array).round(2)


In [22]:
print("=" * 60)
print("DEMAND & PRICE ENGINEERING COMPLETE")
print("=" * 60)
print(f"\nDemand Statistics:")
print(f"  Range: {df['demand'].min()} to {df['demand'].max()} units")
print(f"  Mean: {df['demand'].mean():.2f}")
print(f"  Std Dev: {df['demand'].std():.2f}")

print(f"\nPrice Statistics:")
print(f"  Range: ${df['price'].min():.2f} to ${df['price'].max():.2f}")
print(f"  Mean: ${df['price'].mean():.2f}")
print(f"  Std Dev: ${df['price'].std():.2f}")

print(f"\nTier Comparison:")
tier1_avg_demand = df[df['city_tier'] == 'Tier 1 (Major)']['demand'].mean()
tier2_avg_demand = df[df['city_tier'] == 'Tier 2 (Mid-level)']['demand'].mean()
ratio = tier1_avg_demand / tier2_avg_demand

print(f"  Tier 1 (Major) average demand: {tier1_avg_demand:.2f} units")
print(f"  Tier 2 (Mid-level) average demand: {tier2_avg_demand:.2f} units")
print(f"  Ratio (Tier 1 / Tier 2): {ratio:.2f}x")

price_demand_corr = df['price'].corr(df['demand'])
print(f"\nPrice-Demand Correlation: {price_demand_corr:.3f}")
print("  (0.0 = no correlation, 0.5+ = moderate positive correlation)")
print("=" * 60)

DEMAND & PRICE ENGINEERING COMPLETE

Demand Statistics:
  Range: 0 to 867 units
  Mean: 77.41
  Std Dev: 88.36

Price Statistics:
  Range: $13.65 to $2371.36
  Mean: $215.32
  Std Dev: $297.18

Tier Comparison:
  Tier 1 (Major) average demand: 179.29 units
  Tier 2 (Mid-level) average demand: 29.47 units
  Ratio (Tier 1 / Tier 2): 6.08x

Price-Demand Correlation: 0.210
  (0.0 = no correlation, 0.5+ = moderate positive correlation)


In [23]:
# Replace old store_id with new location-based store_id
old_to_new_rename = old_to_new_df.rename(columns={'old_store_id': 'store_id'})
df = df.merge(old_to_new_rename[['store_id', 'new_store_id']], on='store_id', how='left')
df['store_id'] = df['new_store_id']
df = df.drop(columns=['new_store_id'])

In [24]:
final_columns = [
    'store_id', 'item_id', 'category', 'subcategory', 'price',
    'demand', 'country', 'city_tier', 'date',
    'year', 'month', 'day_of_week', 'is_weekend'
]
df = df[final_columns]

In [25]:
print("Processing Complete!")
display(df.sample(10))

Processing Complete!


,store_id,item_id,category,subcategory,price,demand,country,city_tier,date,year,month,day_of_week,is_weekend
682955,208,item_25,Fashion,Shoes,146.97,255,South Africa,Tier 1 (Major),2019-02-01,2019,2,4,0
2627938,363,item_40,Home,Furniture,221.15,23,Ghana,Tier 2 (Mid-level),2019-11-21,2019,11,3,0
1739899,387,item_3,Fashion,Shoes,112.09,65,Ghana,Tier 2 (Mid-level),2023-03-28,2023,3,1,0
2587928,363,item_18,Fashion,Shoes,76.04,38,Ghana,Tier 2 (Mid-level),2020-05-01,2020,5,4,0
3757093,77,item_8,Fashion,Shoes,33.59,6,Nigeria,Tier 2 (Mid-level),2021-10-08,2021,10,4,0
630706,277,item_46,Electronics,Accessories,26.42,25,South Africa,Tier 2 (Mid-level),2021-01-06,2021,1,2,0
2541930,206,item_43,Home,Appliances,143.77,37,Kenya,Tier 2 (Mid-level),2019-05-19,2019,5,6,1
3235455,476,item_22,Fashion,Clothing,21.95,23,Egypt,Tier 2 (Mid-level),2023-05-29,2023,5,0,0
2828593,306,item_50,Home,Furniture,215.64,26,South Africa,Tier 2 (Mid-level),2019-04-30,2019,4,1,0
537078,102,item_45,Home,Furniture,488.93,100,Kenya,Tier 1 (Major),2019-08-23,2019,8,4,0


In [26]:
# ==========================================
# Save the Final Processed Dataset
# ==========================================
# Define the path to your processed data folder
processed_file_path = '../data/processed/cleaned_retail_sales.csv'

# Save the DataFrame to CSV
# We use index=False so pandas doesn't accidentally save the row numbers as a new column
df.to_csv(processed_file_path, index=False)

print(f"Success! {len(df)} rows of enriched, cleaned data successfully saved to: {processed_file_path}")

Success! 4565000 rows of enriched, cleaned data successfully saved to: ../data/processed/cleaned_retail_sales.csv
